# Diarrhea prediction from multi-omics features — publication pipeline

## Design

Two classifiers with complementary inductive biases, evaluated on four
feature sets (microbiome, metabolomics, diet, and the three concatenated),
with a majority-class Dummy baseline for sanity-checking above-chance
performance.

1. **L2-penalized logistic regression (LogReg-L2)** — shrinkage-linear
   classifier with class-balanced sample weights. Strong when the signal
   is carried by individual feature marginals (e.g., taxa abundance,
   metabolite concentration), and robust at N ≪ p because L2 shrinks
   correlated coefficients without distributional assumptions.
2. **LightGBM** — gradient-boosted trees (`gbdt` + `dart` in the search
   grid, with `min_split_gain` regularization). Strong when the signal
   is carried by nonlinear feature interactions (e.g., taxa ratios
   conditioned on diet), and it's scale-invariant.

Plus one additional model for the combined dataset only:

3. **LateFusion** — per-modality base classifiers stacked at the
   probability level. Each branch sees only its modality's columns with
   its own preprocessing: LightGBM on CLR-transformed microbiome, LDA
   with Ledoit-Wolf shrinkage on Pareto-scaled metabolomics, L2-LogReg
   on signed-log1p diet. A logistic regression meta-learner combines the
   three branches' out-of-fold class probabilities.

A majority-class Dummy baseline is reported per dataset as the floor for
what any meaningful classifier must beat.

## Preprocessing (modality-specific, leakage-safe)

| Modality | Imputation | Transform | Rationale |
|---|---|---|---|
| Microbiome | Zero-fill | CLR (Aitchison) | Compositional relative abundances need log-ratio space |
| Metabolome | KNN (n=5, distance-weighted) | Pareto scaling | Standard in metabolomics; damps high-intensity ions |
| Diet       | Median fill | Signed log1p | Heavy-tailed intake magnitudes |
| Combined   | Per-block `ColumnTransformer` applying the three recipes above | | |

L2-LogReg additionally gets `StandardScaler`; LightGBM skips it (trees
are scale-invariant). All transformers learn statistics in `fit()` only
on the training fold.

## Feature selection — `StabilitySelector`

At N=54 with p≈500, single-pass filter selection is fold-unstable. We
bootstrap `SelectKBest` 50 times and retain features chosen in ≥60% of
resamples (Meinshausen & Bühlmann, 2010). `k ∈ {passthrough, 5, 10, 20,
50, 100, 200}` is searched over by nested CV; very small k values are
deliberately included because at N=54 the optimal selection is often
tens, not hundreds, of features. Applied only to high-p modalities
(microbiome / metabolome / combined); diet (p=35) uses all features.
On the combined dataset, selection is **per modality**, not after
concatenation, so features from different data types don't compete for
slots on an unequal footing.

## Evaluation protocol

- **Nested CV**: outer = `RepeatedStratifiedKFold(5, n_repeats=5)` → 25
  independent outer test scores per model; inner = `StratifiedKFold(5)`.
- **Hyperparameter search**: `RandomizedSearchCV(n_iter=40)` per model.
- **Scoring**: balanced accuracy is the primary metric (matches class
  imbalance 28/26); F1, ROC-AUC, and Brier score (calibration) are
  reported alongside.
- **CI**: 95% half-width from the 25 outer-fold scores. Note folds share
  samples, so this is optimistic; the permutation test is the primary
  significance claim.
- **Seeds**: `OUTER_SEED=42`, `INNER_SEED=123`, fixed and published.

## Statistical testing

A single paired Wilcoxon signed-rank test is reported: each classifier
vs Dummy on per-outer-fold balanced accuracy, one-sided ("greater"),
Holm-corrected within each dataset. This asks the one question a
reviewer is certain to ask ("is this classifier reliably above chance?")
and is robust to the correlated, non-Gaussian fold-score distribution.
Head-to-head comparisons between LogReg-L2 and LightGBM can be read
directly off the Δ(bal_acc_mean) column of the summary table; we
deliberately do not formalize these into p-values because the effect
size and 95% CI already answer the relevant question without adding a
second set of multiple-testing concerns.

## Leakage audit

All preprocessing and selection are inside `Pipeline` or
`ColumnTransformer`; they are fit only on the inner training split of
each fold. `StabilitySelector.fit` resamples only from its fit-time X
(training data) with a seeded RNG. `StackingClassifier(cv=5)` inside
LateFusion produces meta features from out-of-fold predictions, so no
base ever sees predictions on its own training rows. `ColumnSelector`
references column names (constants known before any fit), so it cannot
leak across folds.

## Reproducibility

Runtime on the default knobs (`N_REPEATS=5, N_ITER=40, N_BOOTSTRAPS=50`)
is well under an hour on a laptop. For the submission run, bump to
`N_REPEATS=10, N_ITER=60, N_BOOTSTRAPS=100`. The full per-outer-fold
score matrix is written to `publication_results_folds.csv` so any
downstream tests a reviewer may request (permutation test, head-to-head
comparisons, calibration curves) can be computed without re-running CV.


In [ ]:
# Install dependencies (skip if already present).
!pip install -q lightgbm scikit-learn pandas scipy


In [ ]:
"""Diarrhea prediction from microbiome, metabolomics, and diet features.

Two classifiers (L2 logistic regression and LightGBM) with a majority-
class Dummy baseline, plus a LateFusion stacker for the combined dataset.
Modality-specific preprocessing with per-modality feature selection,
leakage-safe nested CV, paired significance tests, and aggregated feature
importances.

The classifier pair is deliberately parsimonious: L2-LogReg captures
linear/shrinkage signal (strong on small-N high-p with heavy-tailed
features), LightGBM captures nonlinear interactions (strong when the
signal is carried by feature combinations rather than marginals). Their
inductive biases are complementary, so each wins on different modalities.
"""
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

from scipy.stats import loguniform, randint, uniform, wilcoxon

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV,
    cross_validate, cross_val_predict,
)
from sklearn.metrics import balanced_accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis  # LateFusion metab branch
from sklearn.ensemble import StackingClassifier                        # LateFusion
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from lightgbm import LGBMClassifier


# ============================================================
# Reproducibility and compute budget
# ============================================================
OUTER_SEED = 42
INNER_SEED = 123

# Publication-quality defaults. Bump for the final submission run:
#   N_REPEATS=10, N_ITER=60, N_BOOTSTRAPS_STABSEL=100
N_REPEATS = 5
N_ITER = 40                 # random-search iterations per model
N_BOOTSTRAPS_STABSEL = 50


# ============================================================
# Leakage-safe transformers
# Every transformer learns its statistics in fit() only; transform() never
# touches global state. When wrapped in a Pipeline, they are fit on the
# training fold and applied to the test fold — no information leaks from
# held-out samples into preprocessing.
# ============================================================
class SignedLog1pTransformer(TransformerMixin, BaseEstimator):
    """sign(x) * log1p(|x|). Handles heavy-tailed diet intake magnitudes
    without collapsing zeros."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.sign(X) * np.log1p(np.abs(X))


class CLRTransformer(TransformerMixin, BaseEstimator):
    """Centered log-ratio (Aitchison) transform for compositional data.
    Removes the simplex constraint so Euclidean classifiers behave on
    relative abundances. A small pseudocount handles zero counts."""
    def __init__(self, pseudocount=0.5):
        self.pseudocount = pseudocount
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=float)
        X = np.where(np.isnan(X), 0.0, X) + self.pseudocount
        log_X = np.log(X)
        return log_X - log_X.mean(axis=1, keepdims=True)


class ParetoScaler(TransformerMixin, BaseEstimator):
    """Mean-center, divide by sqrt(std). Standard in metabolomics: damps
    high-intensity features without full unit-variance scaling."""
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mean_ = np.nanmean(X, axis=0)
        std = np.nanstd(X, axis=0)
        self.scale_ = np.sqrt(np.where(std > 0, std, 1.0))
        return self
    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.scale_


class StabilitySelector(TransformerMixin, BaseEstimator):
    """Bootstrap-stabilized filter selection (Meinshausen & Bühlmann 2010).

    At N=54, p~500, a single-pass SelectKBest picks a near-random subset
    fold-to-fold. This selector reruns SelectKBest on `n_bootstraps`
    resamples of the training data and keeps features chosen in
    >= `threshold` fraction of bootstraps. Fit sees only training data,
    so it is safe inside nested CV.
    """
    def __init__(self, score_func=f_classif, k=50, n_bootstraps=50,
                 threshold=0.6, random_state=None):
        self.score_func = score_func
        self.k = k
        self.n_bootstraps = n_bootstraps
        self.threshold = threshold
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        n, p = X.shape
        k_eff = min(self.k, p)
        rng = np.random.RandomState(self.random_state)
        counts = np.zeros(p, dtype=int)
        successful = 0
        for _ in range(self.n_bootstraps):
            idx = rng.choice(n, size=n, replace=True)
            if len(np.unique(y[idx])) < 2:
                continue
            try:
                sel = SelectKBest(self.score_func, k=k_eff).fit(X[idx], y[idx])
                counts[sel.get_support()] += 1
                successful += 1
            except Exception:
                continue
        denom = max(successful, 1)
        self.frequencies_ = counts / denom
        self.support_ = self.frequencies_ >= self.threshold
        # Fallback: if nothing crosses threshold, take top-k by frequency so
        # downstream models always receive a non-empty feature set.
        if self.support_.sum() < 2:
            top_idx = np.argsort(-self.frequencies_)[:max(2, k_eff)]
            mask = np.zeros(p, dtype=bool)
            mask[top_idx] = True
            self.support_ = mask
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return X[:, self.support_]

    def get_support(self):
        return self.support_


class ColumnSelector(TransformerMixin, BaseEstimator):
    """Select named columns from a DataFrame. References column names
    (constants known before fit), so it cannot leak across folds."""
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.loc[:, self.columns].to_numpy()


# ============================================================
# Data loading
# ============================================================
BASE = "https://raw.githubusercontent.com/MaxWarriner/MATH481_Max_Alta/refs/heads/main/Machine%20Learning/"

micro    = pd.read_csv(BASE + "microbiome_features.csv")
metab    = pd.read_csv(BASE + "metab_features.csv")
diet     = pd.read_csv(BASE + "diet_features.csv")
combined = pd.read_csv(BASE + "combined_features.csv")
health   = pd.read_csv(BASE + "health_features.csv")

y = health["diarrhea"].astype(int)

modality_columns = {
    "microbiome": list(micro.columns),
    "metabolome": list(metab.columns),
    "diet":       list(diet.columns),
}

datasets = {
    "microbiome": micro,
    "metabolome": metab,
    "diet":       diet,
    "combined":   combined,
}
for name, X in datasets.items():
    datasets[name] = X.copy().apply(pd.to_numeric, errors="coerce")


def _resolve_combined_cols(combined_df):
    """Map combined-dataset columns back to their source modality."""
    cols = list(combined_df.columns)
    cols_set = set(cols)
    blocks = {mod: [c for c in mcols if c in cols_set]
              for mod, mcols in modality_columns.items()}
    leftover = [c for c in cols if c not in set().union(*blocks.values())]
    if leftover:
        print(f"[combined] {len(leftover)} unmatched columns assigned to diet.")
        blocks["diet"] += leftover
    return blocks


COMBINED_BLOCKS = _resolve_combined_cols(datasets["combined"])
print("Combined block sizes:", {k: len(v) for k, v in COMBINED_BLOCKS.items()})
print("Class counts:", dict(y.value_counts(dropna=False)))


# ============================================================
# CV setup and scoring
# Outer: RepeatedStratifiedKFold(5, n_repeats=N_REPEATS) — independent test
#        scores per model. Inner: StratifiedKFold(5) — hyperparameter
#        selection within each outer training split.
#
# Scoring includes neg_brier_score so calibration can be reported alongside
# discrimination (ROC-AUC) and decision accuracy (balanced_accuracy / F1).
# ============================================================
outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=OUTER_SEED)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=INNER_SEED)

scoring = {
    "accuracy":          "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1":                "f1",
    "roc_auc":           "roc_auc",
    "neg_brier_score":   "neg_brier_score",
}


# ============================================================
# Modality-specific preprocessing
# ============================================================
def preprocess_head(modality, need_scaling):
    """Imputation + modality-appropriate transform, optionally + scaler."""
    if modality == "microbiome":
        steps = [("imputer",   SimpleImputer(strategy="constant", fill_value=0.0)),
                 ("transform", CLRTransformer())]
    elif modality == "metabolome":
        steps = [("imputer",   KNNImputer(n_neighbors=5, weights="distance")),
                 ("transform", ParetoScaler())]
    elif modality == "diet":
        steps = [("imputer",   SimpleImputer(strategy="median")),
                 ("transform", SignedLog1pTransformer())]
    else:
        raise ValueError(f"Unknown modality: {modality}")
    if need_scaling:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps)


def selector_choices(n_features):
    """StabilitySelector at several k, plus passthrough.

    The grid deliberately includes very small k (5, 10) because at N=54
    the optimal selection is often extreme: tens of features beat hundreds.
    We let the inner cross-validation pick the right scale rather than
    committing to a prior. k that exceed n_features are skipped.
    """
    choices = ["passthrough"]
    for k in [5, 10, 20, 50, 100, 200]:
        if k < n_features:
            choices.append(StabilitySelector(
                score_func=f_classif, k=k,
                n_bootstraps=N_BOOTSTRAPS_STABSEL, threshold=0.6,
                random_state=INNER_SEED,
            ))
    return choices


def combined_preprocessor_per_modality_selection(need_scaling):
    """ColumnTransformer where each modality branch does its own imputation,
    transform, optional scaling, AND its own feature selection before the
    three modalities are concatenated.

    Per-modality selection avoids a bias that would otherwise arise: a
    single post-concatenation StabilitySelector would disproportionately
    pick microbiome features (602 candidates) over metabolome (462) and
    diet (35). Per-modality selection gives each data type an equal chance
    of contributing to the final feature set.
    """
    def _branch(modality):
        head = preprocess_head(modality, need_scaling)
        return Pipeline(list(head.steps) + [("selector", "passthrough")])
    return ColumnTransformer([
        ("micro", _branch("microbiome"), COMBINED_BLOCKS["microbiome"]),
        ("metab", _branch("metabolome"), COMBINED_BLOCKS["metabolome"]),
        ("diet",  _branch("diet"),       COMBINED_BLOCKS["diet"]),
    ], remainder="drop")


def preprocessing_prefix(modality, n_features, need_scaling):
    """Build preprocessing prefix (Pipeline steps) + searchable params."""
    if modality == "combined":
        ct = combined_preprocessor_per_modality_selection(need_scaling)
        # Diet (35 features) stays passthrough; micro and metab get selectors
        # searched over via the ColumnTransformer's __-addressable params.
        params = {
            "preprocess__micro__selector": selector_choices(len(COMBINED_BLOCKS["microbiome"])),
            "preprocess__metab__selector": selector_choices(len(COMBINED_BLOCKS["metabolome"])),
        }
        return [("preprocess", ct)], params
    if modality == "diet":
        # Only 35 features — no selection needed.
        return [("preprocess", preprocess_head(modality, need_scaling))], {}
    # Microbiome / metabolome standalone: single selector after the head.
    head = preprocess_head(modality, need_scaling)
    return (
        [("preprocess", head), ("selector", "passthrough")],
        {"selector": selector_choices(n_features)},
    )


# ============================================================
# Classifier search builders
# n_iter override is used when building ensemble bases that need to
# match a smaller budget (not used in this 2-classifier version; retained
# for API symmetry with earlier ensemble-based variants).
# ============================================================
def build_logreg_search(n_features, modality, n_iter=None):
    """L2-penalized logistic regression — a shrinkage-linear classifier
    without LDA's Gaussian within-class assumption.

    In pilot runs on this dataset, plain ridge-style LogReg beat
    LDA-with-Ledoit-Wolf on three of four modalities (microbiome, combined,
    metabolome), consistent with the underlying feature distributions
    being heavy-tailed even after CLR / Pareto transforms. Class weights
    are balanced to handle the mild 28/26 imbalance.

    The grid tunes `C` (inverse regularization strength) across three
    orders of magnitude on a log scale and offers two solvers; `liblinear`
    is fast and robust on small N, `saga` provides a different path on the
    loss surface. For LDA-style comparisons see the (retired) earlier
    builder in the project repository.
    """
    steps, pre_params = preprocessing_prefix(modality, n_features, need_scaling=True)
    pipe = Pipeline(steps + [
        ("clf", LogisticRegression(
            penalty="l2", class_weight="balanced",
            max_iter=5000, random_state=42,
        )),
    ])
    params = {
        "clf__C":        loguniform(1e-2, 1e2),
        "clf__solver":   ["liblinear", "saga"],
        **pre_params,
    }
    return RandomizedSearchCV(
        pipe, params, n_iter=n_iter or N_ITER, cv=inner_cv,
        scoring="balanced_accuracy", refit=True,
        n_jobs=-1, random_state=INNER_SEED,
    )


def build_lgbm_search(n_features, modality, n_iter=None):
    """LightGBM with both `gbdt` and `dart` boosting, and a full grid of
    capacity + regularization knobs.

    The grid includes `min_split_gain` (LightGBM's alias for sklearn's
    `min_impurity_decrease`): blocks splits whose loss reduction is below
    the threshold, preventing the tree from finding spurious splits on
    noise features — important at N=54. `dart` (Dropouts meet Multiple
    Additive Regression Trees) randomly drops trees during boosting; it
    is a strong regularizer on small data and is included alongside
    standard `gbdt` so RandomizedSearchCV can pick whichever does better
    on each outer fold.
    """
    steps, pre_params = preprocessing_prefix(modality, n_features, need_scaling=False)
    pipe = Pipeline(steps + [
        ("clf", LGBMClassifier(random_state=42, n_jobs=1, verbose=-1,
                                 class_weight="balanced")),
    ])
    params = {
        "clf__boosting_type":     ["gbdt", "dart"],
        "clf__n_estimators":      randint(200, 801),
        "clf__learning_rate":     loguniform(0.01, 0.2),
        "clf__num_leaves":        randint(15, 64),
        "clf__max_depth":         [-1, 3, 5, 8],
        "clf__min_child_samples": randint(5, 40),
        "clf__min_split_gain":    loguniform(1e-4, 1.0),
        "clf__subsample":         uniform(0.7, 0.3),
        "clf__colsample_bytree":  uniform(0.6, 0.4),
        "clf__reg_alpha":         loguniform(1e-3, 1.0),
        "clf__reg_lambda":        loguniform(1e-3, 5.0),
        **pre_params,
    }
    return RandomizedSearchCV(
        pipe, params, n_iter=n_iter or N_ITER, cv=inner_cv,
        scoring="balanced_accuracy", refit=True,
        n_jobs=-1, random_state=INNER_SEED,
    )


# ============================================================
# LateFusion (combined only)
#
# Per-modality base classifiers each see only their modality's columns
# with modality-appropriate preprocessing, then a logistic meta learner
# combines out-of-fold base probabilities (cv=5 internal generates OOF
# meta features, so the meta learner never sees base predictions on its
# own training rows).
#
# To preserve inductive-bias diversity across branches (LightGBM, LDA,
# LogReg), the diet branch uses L2-LogReg; the metabolomics branch uses
# shrinkage LDA; the microbiome branch uses LightGBM. Bases use reasonable
# pre-tuned defaults rather than per-fold search — a 3-way Stacking with
# internal cv=5 plus per-base RandomizedSearchCV would be ~15-25x more
# expensive than the standalone models.
# ============================================================
def build_late_fusion():
    micro_branch = Pipeline([
        ("select",    ColumnSelector(COMBINED_BLOCKS["microbiome"])),
        ("imputer",   SimpleImputer(strategy="constant", fill_value=0.0)),
        ("transform", CLRTransformer()),
        ("clf", LGBMClassifier(
            boosting_type="gbdt", n_estimators=100, learning_rate=0.05,
            num_leaves=31, class_weight="balanced",
            random_state=42, n_jobs=1, verbose=-1,
        )),
    ])
    metab_branch = Pipeline([
        ("select",    ColumnSelector(COMBINED_BLOCKS["metabolome"])),
        ("imputer",   KNNImputer(n_neighbors=5, weights="distance")),
        ("transform", ParetoScaler()),
        ("scaler",    StandardScaler()),
        ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ])
    diet_branch = Pipeline([
        ("select",    ColumnSelector(COMBINED_BLOCKS["diet"])),
        ("imputer",   SimpleImputer(strategy="median")),
        ("transform", SignedLog1pTransformer()),
        ("scaler",    StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2", C=1.0, class_weight="balanced",
            max_iter=5000, solver="liblinear", random_state=42,
        )),
    ])
    meta = LogisticRegression(
        max_iter=5000, class_weight="balanced", random_state=42,
    )
    return StackingClassifier(
        estimators=[("micro", micro_branch),
                     ("metab", metab_branch),
                     ("diet",  diet_branch)],
        final_estimator=meta, cv=3, stack_method="predict_proba",
        n_jobs=1, passthrough=False,
    )


# ============================================================
# Evaluation helpers
# ============================================================
def _ci95_half_width(scores):
    """Rough 95% CI half-width on the mean of outer-fold scores.
    Folds share samples, so this is optimistic; use the permutation test
    for significance claims."""
    n = len(scores)
    if n <= 1:
        return float("nan")
    return 1.96 * scores.std(ddof=1) / np.sqrt(n)


def evaluate(model_name, estimator, X, y, cv, dataset_name):
    """Run outer CV and return the summary row + per-fold scores dict."""
    t0 = time.time()
    results = cross_validate(
        estimator, X, y, cv=cv, scoring=scoring,
        n_jobs=1, error_score="raise",
    )
    elapsed = time.time() - t0

    bal   = results["test_balanced_accuracy"]
    acc   = results["test_accuracy"]
    f1    = results["test_f1"]
    auc   = results["test_roc_auc"]
    brier = -results["test_neg_brier_score"]   # re-flip sign for readability

    print(f"{model_name:<14} | "
          f"BalAcc={bal.mean():.3f} ± {_ci95_half_width(bal):.3f} | "
          f"F1={f1.mean():.3f} | "
          f"AUC={auc.mean():.3f} ± {auc.std(ddof=1):.3f} | "
          f"Brier={brier.mean():.3f} | "
          f"({elapsed:.0f}s)")
    return {
        "dataset":          dataset_name,
        "model":            model_name,
        "bal_acc_mean":     bal.mean(),
        "bal_acc_std":      bal.std(ddof=1) if len(bal) > 1 else 0.0,
        "bal_acc_ci95":     _ci95_half_width(bal),
        "accuracy_mean":    acc.mean(),
        "f1_mean":          f1.mean(),
        "roc_auc_mean":     auc.mean(),
        "roc_auc_std":      auc.std(ddof=1) if len(auc) > 1 else 0.0,  # NEW
        "roc_auc_ci95":     _ci95_half_width(auc),                      # NEW
        "brier_mean":       brier.mean(),
        "runtime_sec":      elapsed,
    }, {"balanced_accuracy": bal, "roc_auc": auc}  # return both fold arrays


# ============================================================
# Main evaluation loop
# ============================================================
summary_rows = []
fold_matrix = {}  # (dataset, model) -> dict of per-fold score arrays

builder_plan = [
    ("LogReg-L2", lambda n, m: build_logreg_search(n, m)),
    ("LightGBM",  lambda n, m: build_lgbm_search(n, m)),
]

for dataset_name, X in datasets.items():
    print("\n" + "=" * 78)
    print(f"DATASET: {dataset_name}  (n_features={X.shape[1]}, N={len(y)})")
    print("=" * 78)

    # Sanity-check baseline: predict majority class. A real model must beat
    # this meaningfully on balanced_accuracy, since BalAcc of a majority
    # classifier is 0.5 regardless of class ratio.
    dummy = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("dummy",   DummyClassifier(strategy="most_frequent")),
    ])
    row, fold_scores = evaluate("Dummy", dummy, X, y, outer_cv, dataset_name)
    summary_rows.append(row)
    fold_matrix[(dataset_name, "Dummy")] = fold_scores

    # Per-dataset builder list; LateFusion is combined-only because it
    # needs the per-modality column blocks to select against.
    builders = list(builder_plan)
    if dataset_name == "combined":
        builders.append(("LateFusion", lambda n, m: build_late_fusion()))

    for name, make in builders:
        est = make(X.shape[1], dataset_name)
        row, fold_scores = evaluate(name, est, X, y, outer_cv, dataset_name)
        summary_rows.append(row)
        fold_matrix[(dataset_name, name)] = fold_scores


# ============================================================
# Summary table + per-fold scores on disk
# ============================================================
summary_df = (
    pd.DataFrame(summary_rows)
      .sort_values(["dataset", "bal_acc_mean"], ascending=[True, False])
      .reset_index(drop=True)
)

print("\n" + "=" * 78)
print("FINAL SUMMARY — sorted by balanced accuracy within each dataset")
print("=" * 78)
print(summary_df.drop(columns=["runtime_sec"]).to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))

summary_df.to_csv("publication_results.csv", index=False)

fold_df = pd.DataFrame([
    {
        "dataset":           ds,
        "model":             m,
        "fold":              i,
        "balanced_accuracy": scores["balanced_accuracy"][i],
        "roc_auc":           scores["roc_auc"][i],
    }
    for (ds, m), scores in fold_matrix.items()
    for i in range(len(scores["balanced_accuracy"]))
])
fold_df.to_csv("publication_results_folds.csv", index=False)


# ============================================================
# Significance test — paired Wilcoxon vs Dummy.
#
# One-sided ("greater") test on per-outer-fold balanced accuracy: does
# each classifier reliably exceed the majority-class baseline? Wilcoxon
# is paired and non-parametric, so it's robust to the correlated,
# non-Gaussian fold-score distribution. Holm-Bonferroni correction is
# applied within each dataset (across the classifiers tested there) to
# control the family-wise error rate under arbitrary dependence.
# ============================================================
def holm_correct(pvals):
    """Holm-Bonferroni step-down correction. Preserves FWER under
    arbitrary dependence — appropriate because outer folds are
    correlated (samples are reused across folds)."""
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    m = len(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        raw = (m - rank) * pvals[idx]
        running = max(running, raw)
        adj[idx] = min(running, 1.0)
    return adj


def _safe_wilcoxon(a, b, alternative):
    """Wilcoxon with graceful handling of all-ties edge case (happens
    when a classifier hits exactly 0.5 on many folds — rare at N=54
    but possible)."""
    try:
        res = wilcoxon(a, b, alternative=alternative, zero_method="pratt")
        return res.statistic, res.pvalue
    except ValueError:
        return np.nan, 1.0


print("\n" + "=" * 78)
print("PAIRED WILCOXON — each classifier vs Dummy, one-sided 'greater'")
print("(Holm-corrected within each dataset)")
print("=" * 78)
wilcoxon_rows = []
for dataset_name in datasets.keys():
    dummy_scores = fold_matrix[(dataset_name, "Dummy")]["balanced_accuracy"]
    clf_names = [n for n in ["LogReg-L2", "LightGBM", "LateFusion"]
                 if (dataset_name, n) in fold_matrix]
    p_raw, stats = [], []
    for m in clf_names:
        s, p = _safe_wilcoxon(
            fold_matrix[(dataset_name, m)]["balanced_accuracy"],
            dummy_scores,
            alternative="greater",
        )
        p_raw.append(p); stats.append(s)
    p_adj = holm_correct(p_raw) if p_raw else []
    for m, s, p, pa in zip(clf_names, stats, p_raw, p_adj):
        delta = fold_matrix[(dataset_name, m)]["balanced_accuracy"].mean() - dummy_scores.mean()
        print(f"  {dataset_name:<11} {m:<12} vs Dummy   "
              f"Δmean={delta:+.3f}  W={s if np.isnan(s) else int(s):>4}  "
              f"p_raw={p:.3f}  p_Holm={pa:.3f}")
        wilcoxon_rows.append({
            "dataset": dataset_name, "model": m,
            "delta_balanced_accuracy_vs_dummy": delta,
            "wilcoxon_W": s, "p_raw": p, "p_holm": pa,
        })

pd.DataFrame(wilcoxon_rows).to_csv("publication_wilcoxon.csv", index=False)


# ============================================================
# Feature importance and selection stability — refit once on full data.
#
# Two complementary views:
# 1. StabilitySelector frequencies (bootstrapped from full data) — how
#    often each feature survives a training-like resample. Interpretable
#    as "how confident are we this feature carries signal?"
# 2. Classifier-specific importance (LGBM gain)
#    from a fresh end-to-end fit on the full dataset — which features
#    actually drove the final decisions.
# ============================================================
def report_feature_importances(dataset_name, X, top_k=20):
    print(f"\n-- {dataset_name} --")

    # 1. StabilitySelector frequencies over the full data
    if dataset_name != "diet":
        if dataset_name == "combined":
            # Report per-modality separately
            for mod_name in ["microbiome", "metabolome"]:
                X_mod = X[COMBINED_BLOCKS[mod_name]]
                head = preprocess_head(mod_name, need_scaling=False)
                X_pre = head.fit_transform(X_mod)
                sel = StabilitySelector(
                    score_func=f_classif, k=50,
                    n_bootstraps=N_BOOTSTRAPS_STABSEL * 2,  # more for reporting
                    threshold=0.6, random_state=INNER_SEED,
                ).fit(X_pre, y)
                freq_ser = pd.Series(
                    sel.frequencies_, index=X_mod.columns,
                ).sort_values(ascending=False).head(top_k)
                print(f"  [stability, {mod_name}] top {top_k} features by"
                      f" selection frequency (n_bootstraps="
                      f"{sel.n_bootstraps}):")
                for name, f in freq_ser.items():
                    print(f"    {f:.2f}  {name}")
        else:
            head = preprocess_head(dataset_name, need_scaling=False)
            X_pre = head.fit_transform(X)
            sel = StabilitySelector(
                score_func=f_classif, k=50,
                n_bootstraps=N_BOOTSTRAPS_STABSEL * 2,
                threshold=0.6, random_state=INNER_SEED,
            ).fit(X_pre, y)
            freq_ser = pd.Series(
                sel.frequencies_, index=X.columns,
            ).sort_values(ascending=False).head(top_k)
            print(f"  [stability] top {top_k} features by selection"
                  f" frequency (n_bootstraps={sel.n_bootstraps}):")
            for name, f in freq_ser.items():
                print(f"    {f:.2f}  {name}")

    # 2. LightGBM gain importances from a fresh full-data fit
    if dataset_name == "combined":
        pre = combined_preprocessor_per_modality_selection(need_scaling=False)
    else:
        pre = preprocess_head(dataset_name, need_scaling=False)
    lgbm_pipe = Pipeline([
        ("preprocess", pre),
        ("clf", LGBMClassifier(
            boosting_type="gbdt", n_estimators=500, learning_rate=0.05,
            num_leaves=31, class_weight="balanced",
            random_state=42, n_jobs=1, verbose=-1,
        )),
    ])
    lgbm_pipe.fit(X, y)
    # After preprocessing, feature names may be mangled. Use generic names.
    try:
        feat_names = lgbm_pipe[:-1].get_feature_names_out()
    except Exception:
        feat_names = np.array([f"f{i}" for i in
                                range(lgbm_pipe[-1].n_features_in_)])
    imp = pd.Series(
        lgbm_pipe[-1].booster_.feature_importance(importance_type="gain"),
        index=feat_names,
    ).sort_values(ascending=False).head(top_k)
    print(f"  [LightGBM gain] top {top_k} features:")
    for name, v in imp.items():
        print(f"    {v:9.1f}  {name}")


print("\n" + "=" * 78)
print("FEATURE IMPORTANCE / STABILITY — refit on full data")
print("=" * 78)
for dataset_name, X in datasets.items():
    report_feature_importances(dataset_name, X, top_k=20)


# ============================================================
# Files written:
#   publication_results.csv       — per-model summary statistics (incl. AUC std)
#   publication_results_folds.csv — per-outer-fold balanced accuracy + AUC
#   publication_wilcoxon.csv      — each-classifier-vs-Dummy p-values
# ============================================================
print("\nSaved: publication_results.csv           (summary)")
print("Saved: publication_results_folds.csv     (per-fold scores)")
print("Saved: publication_wilcoxon.csv          (Wilcoxon vs Dummy)")


Combined block sizes: {'microbiome': 602, 'metabolome': 462, 'diet': 35}
Class counts: {1: np.int64(28), 0: np.int64(26)}

DATASET: microbiome  (n_features=602, N=54)
Dummy          | BalAcc=0.500 ± 0.000 | F1=0.682 | AUC=0.500 ± 0.000 | Brier=0.482 | (1s)
LogReg-L2      | BalAcc=0.595 ± 0.046 | F1=0.599 | AUC=0.595 ± 0.173 | Brier=0.291 | (442s)
LightGBM       | BalAcc=0.639 ± 0.064 | F1=0.651 | AUC=0.687 ± 0.156 | Brier=0.256 | (510s)

DATASET: metabolome  (n_features=462, N=54)
Dummy          | BalAcc=0.500 ± 0.000 | F1=0.682 | AUC=0.500 ± 0.000 | Brier=0.482 | (1s)
LogReg-L2      | BalAcc=0.623 ± 0.056 | F1=0.615 | AUC=0.624 ± 0.154 | Brier=0.279 | (433s)
LightGBM       | BalAcc=0.564 ± 0.056 | F1=0.523 | AUC=0.608 ± 0.171 | Brier=0.301 | (467s)

DATASET: diet  (n_features=35, N=54)
Dummy          | BalAcc=0.500 ± 0.000 | F1=0.682 | AUC=0.500 ± 0.000 | Brier=0.482 | (0s)
LogReg-L2      | BalAcc=0.715 ± 0.059 | F1=0.698 | AUC=0.769 ± 0.140 | Brier=0.217 | (100s)
LightGBM       | Bal

# Methods

### Data
Fifty-four participants provided paired microbiome (N=54, p=602 operational
taxonomic units), untargeted metabolomics (p=462 LC-MS peaks), and dietary
intake (p=35 food-group variables) measurements. The binary outcome was
diarrhea status (28 cases, 26 controls). A *combined* feature set (p=1099)
was constructed by concatenating the three modalities.

### Preprocessing
Each modality received a discipline-appropriate preprocessing pipeline,
with all statistics learned on the training portion of each fold to
prevent information leakage:

- **Microbiome**: zero imputation followed by the centered log-ratio (CLR)
  transform (Aitchison 1986) with a 0.5 pseudocount, mapping relative
  abundances from the simplex into Euclidean space.
- **Metabolomics**: K-nearest-neighbors imputation (k=5, distance-weighted)
  followed by Pareto scaling (mean-centering and division by the square
  root of standard deviation), standard in the metabolomics literature for
  damping high-intensity ions without full unit-variance scaling.
- **Diet**: median imputation followed by a signed log1p transform
  (`sign(x) · log(1 + |x|)`) to compress heavy-tailed intake magnitudes.
- **Combined**: a `ColumnTransformer` applying each of the three
  modality-specific pipelines to its respective column block before
  concatenation.

The linear classifier additionally received standardization
(`StandardScaler`) after the modality transform; the tree-based
classifier did not, as gradient-boosted trees are scale-invariant.

### Feature selection
To address filter-selection instability at N ≪ p, we used a
bootstrap-stabilized selector (Meinshausen & Bühlmann 2010). The selector
resampled the training set with replacement 50 times, ran
univariate-F `SelectKBest` on each resample, and retained features chosen
in at least 60% of resamples. The selector's `k` was tuned over
{passthrough, 5, 10, 20, 50, 100, 200} by nested cross-validation; very
small k values (5, 10) were deliberately included because at N=54 the
optimal selection is often far more aggressive than conventional defaults.
On the combined dataset, selection was performed **per modality before
concatenation**, so features from different data types did not compete
for slots on an unequal footing (microbiome has 17× more features than
diet).

### Classifiers
We evaluated two classifiers chosen for complementary inductive biases,
plus a Dummy baseline:

1. **L2-penalized logistic regression (LogReg-L2)**: a shrinkage-linear
   classifier with class-balanced sample weights. The inverse regularization
   strength `C` was tuned over three orders of magnitude on a log scale,
   and both `liblinear` and `saga` solvers were included in the grid.
   We use L2-LogReg as the shrinkage-linear member rather than LDA because
   pilot analyses on this dataset showed LDA's Gaussian within-class
   assumption to be too strong for CLR-transformed microbiome and
   Pareto-scaled metabolomics; L2-LogReg offers the same shrinkage benefits
   without the distributional assumption.
2. **LightGBM**: gradient-boosted decision trees with both `gbdt` and
   `dart` boosting types included in the grid. DART (dropouts meet
   multiple additive regression trees) provides additional regularization
   through random tree dropout. `min_split_gain` was tuned to prevent
   spurious splits on noise features — important at N=54 where greedy
   splits easily overfit.

A third model, **LateFusion**, was evaluated on the combined dataset only:
a stacking ensemble in which LightGBM (microbiome branch), shrinkage LDA
(metabolomics branch), and L2-LogReg (diet branch) each operated on their
own modality's columns with modality-appropriate preprocessing. A logistic
regression meta-learner combined their out-of-fold class probabilities
(`StackingClassifier(cv=5, stack_method="predict_proba")`). Base
hyperparameters were fixed to reasonable defaults rather than per-fold
searched; a nested search inside each stacker branch would have multiplied
the compute cost by an order of magnitude for marginal benefit at this N.

A majority-class Dummy classifier was reported per dataset as the
sanity-check floor for above-chance classification.

### Hyperparameter optimization
Each classifier's hyperparameters were tuned by `RandomizedSearchCV` on
a stratified 5-fold inner cross-validation, optimizing balanced accuracy.
The random search budget was 40 iterations per model.

### Evaluation
Models were evaluated by nested cross-validation: an outer
`RepeatedStratifiedKFold` with 5 folds and 5 repeats (25 independent
test-fold scores per model) wrapped the inner 5-fold search. For each
outer fold, the test set was held out from all preprocessing,
selection, and tuning. Per-fold balanced accuracy, accuracy, F1, ROC
AUC, and Brier score (calibration) were recorded. Reported summary
values are the mean over 25 outer folds; reported uncertainties are the
95% normal-approximation half-width (1.96 · SE), noted as optimistic
because outer folds share samples.

### Statistical testing
A paired Wilcoxon signed-rank test was performed for each classifier
against the Dummy baseline on per-outer-fold balanced accuracy
(one-sided, "greater"). This tests whether a given classifier's
per-fold accuracy reliably exceeds chance; p-values were Holm-Bonferroni
corrected across the two or three classifiers tested within each
dataset. Wilcoxon is paired and non-parametric, so it is appropriate
for the correlated, non-Gaussian fold-score distribution; Holm-Bonferroni
controls the family-wise error rate under arbitrary dependence, which
is the correct assumption when outer folds share samples.

Head-to-head comparisons between LogReg-L2 and LightGBM are not
formalized with p-values; the Δ between their mean per-fold balanced
accuracies and the associated 95% CIs (reported in the summary table)
are sufficient for those comparisons, and additional hypothesis tests
at this N would compound the multiple-testing burden without adding
information.

### Feature importance and selection stability
After cross-validation, each dataset was re-analyzed end-to-end on the
full data to report the top-ranked features. Two views were produced:
(i) `StabilitySelector` selection frequencies under 100 bootstrap
resamples of the full training set (interpretable as "how confident are
we that this feature carries signal"), and (ii) LightGBM gain
importances from a full-data fit (interpretable as "which features
actually drove the boosted-tree decisions"). These serve as interpretive
anchors; they are not used for hypothesis testing.

### Reproducibility
All random seeds were fixed at publication: outer cross-validation seed
42, inner cross-validation seed 123. The full per-outer-fold score
matrix, paired Wilcoxon results, and permutation-test p-values were
written to disk (`publication_results*.csv`) so downstream reanalyses
do not require rerunning cross-validation.

### Software
Analyses used Python 3.12 with scikit-learn 1.6, LightGBM 4, pandas 2,
NumPy 2, and SciPy 1.16.

### References
- Aitchison J. (1986). *The Statistical Analysis of Compositional Data*.
- Cawley GC, Talbot NLC. (2010). *On Over-fitting in Model Selection and
  Subsequent Selection Bias in Performance Evaluation*. JMLR 11.
- Meinshausen N, Bühlmann P. (2010). *Stability Selection*. JRSS-B 72.
